# Assignment 4 — Automated Insight Generation

## Auto-Analytics Engine for District-Level Healthcare Performance

This notebook demonstrates the end-to-end analytical pipeline:
- **Part A**: Data Loading & Validation
- **Part B**: Trend Detection (with configurable threshold)
- **Part C**: Outlier Detection (IQR & Z-score rules)
- **Part D**: Correlation Matrix & Fragility Analysis
- **Part E**: Automated Dynamic Insight Generation & Severity Scoring
- **Part F**: Visualizations

In [ ]:
import os
import pandas as pd
import numpy as np
from src.data_loader import load_dataset, validate_schema, get_missing_values_report, print_dataset_summary
from src.analyzer import detect_trends, detect_outliers_iqr, detect_outliers_zscore, compute_correlation_matrix, flag_high_correlations
from src.insight_generator import generate_all_insights

### Part A: Data Loading & Validation
Ingest `data/healthcare_data.csv`, inspect head, info, and missing values.

In [ ]:
df = load_dataset('data/healthcare_data.csv')
print('Schema Valid:', validate_schema(df))
print_dataset_summary(df)

### Part B: Trend Detection
For each (district, indicator) pair, compute:
$$\text{pct\_change} = \frac{\text{current} - \text{previous}}{\text{previous}} \times 100$$
Flag as significant if $|\text{pct\_change}| \ge 10\%$.

In [ ]:
trends_df = detect_trends(df, threshold_pct=10.0)
significant_trends = trends_df[trends_df['is_significant']]
significant_trends[['district', 'indicator', 'prev_val', 'current_val', 'change_pct']]

### Part C: Outlier Detection
IQR rule ($Q_1 - 1.5\times IQR$ / $Q_3 + 1.5\times IQR$) and Z-score rule ($|z| \ge 2.5$).

In [ ]:
iqr_outliers = detect_outliers_iqr(df, iqr_multiplier=1.5)
print('IQR Outliers:')
display(iqr_outliers[['district', 'indicator', 'month', 'value', 'lower_bound', 'upper_bound']])

z_outliers = detect_outliers_zscore(df, z_threshold=2.5)
print('\nZ-Score Outliers:')
display(z_outliers[['district', 'indicator', 'month', 'value', 'benchmark_mean', 'z_score']])

### Part D: Pearson Correlation Matrix
> **Limitation Note**: With only 2 months $\times$ 6 districts (12 rows), true correlation is statistically fragile and sensitive to individual outliers (e.g., Mehsana).

In [ ]:
corr_matrix = compute_correlation_matrix(df)
print('Pearson Correlation Matrix:')
display(corr_matrix)

flagged_pairs = flag_high_correlations(corr_matrix, threshold=0.70)
print('\nFlagged Highly Correlated Pairs (|r| >= 0.70):')
display(flagged_pairs[['indicator_pair', 'correlation_r', 'abs_r']])

### Part E: Automated Dynamic Insights & Severity Evaluation

In [ ]:
insights_df = generate_all_insights(df, trend_threshold_pct=10.0, corr_threshold=0.70)
display(insights_df)
insights_df.to_csv('sample_insights.csv', index=False)
insights_df.to_json('sample_insights.json', orient='records', indent=2)